# Drone Sound Detection Training

This notebook trains a drone sound detection model using the [Drone-detection-dataset](https://github.com/DroneDetectionThesis/Drone-detection-dataset).

## Goal
Build a binary classifier that outputs a value between 0 and 1:
- **1**: Drone detected
- **0**: Not a drone (Background or Helicopter)

## Environment
This notebook is designed to run in **Google Colab** using the session disk for data storage.

## 1. Setup and Data Acquisition

In [ ]:
# Clone the repository to the session disk
!git clone https://github.com/DroneDetectionThesis/Drone-detection-dataset.git /content/drone_dataset

In [ ]:
import os
import numpy as np
import pandas as pd
import librosa
import librosa.display
import matplotlib.pyplot as plt
from tqdm import tqdm
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

## 2. Dataset Preparation

We will scan the `Data/Audio` directory and label the files.

In [ ]:
AUDIO_DIR = '/content/drone_dataset/Data/Audio'

def get_dataset_files(audio_dir):
    data = []
    for root, dirs, files in os.walk(audio_dir):
        for file in files:
            if file.endswith(('.wav', '.mp3', '.m4a')):
                path = os.path.join(root, file)
                filename = file.lower()
                
                # Labeling logic based on filename keywords
                if 'drone' in filename:
                    label = 1
                else:
                    # Helicopter and Background are 'Not Drone'
                    label = 0
                
                data.append({'path': path, 'label': label, 'name': file})
    return pd.DataFrame(data)

df = get_dataset_files(AUDIO_DIR)
print(f"Total files found: {len(df)}")
print(df['label'].value_counts())
df.head()

## 3. Preprocessing and Feature Extraction

We will extract Mel-spectrograms from the audio clips. We'll split long clips into smaller chunks (e.g., 2 seconds) to increase the number of training samples.

In [ ]:
SAMPLE_RATE = 22050
DURATION = 2 # seconds
SAMPLES_PER_TRACK = SAMPLE_RATE * DURATION

def extract_features(df, n_mels=128):
    X = []
    y = []
    
    for idx, row in tqdm(df.iterrows(), total=len(df)):
        try:
            # Load audio file
            signal, sr = librosa.load(row['path'], sr=SAMPLE_RATE)
            
            # Process in chunks of DURATION
            for d in range(0, len(signal) - SAMPLES_PER_TRACK, SAMPLES_PER_TRACK // 2): # 50% overlap
                chunk = signal[d : d + SAMPLES_PER_TRACK]
                
                # Extract Mel Spectrogram
                mel_spec = librosa.feature.melspectrogram(y=chunk, sr=sr, n_mels=n_mels)
                log_mel_spec = librosa.power_to_db(mel_spec, ref=np.max)
                
                # Normalize
                log_mel_spec = (log_mel_spec - log_mel_spec.min()) / (log_mel_spec.max() - log_mel_spec.min())
                
                X.append(log_mel_spec[..., np.newaxis])
                y.append(row['label'])
        except Exception as e:
            print(f"Error processing {row['path']}: {e}")
            
    return np.array(X), np.array(y)

print("Extracting features... This may take a few minutes.")
X, y = extract_features(df)
print(f"Dataset shape: X={X.shape}, y={y.shape}")

## 4. Model Definition

A simple CNN architecture for binary classification.

In [ ]:
def create_model(input_shape):
    model = models.Sequential([
        layers.Conv2D(32, (3, 3), activation='relu', input_shape=input_shape),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.Flatten(),
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(1, activation='sigmoid') # Binary output 0-1
    ])
    
    model.compile(optimizer='adam',
                  loss='binary_crossentropy',
                  metrics=['accuracy'])
    return model

input_shape = X.shape[1:]
model = create_model(input_shape)
model.summary()

## 5. Training

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

history = model.fit(X_train, y_train, 
                    epochs=20, 
                    batch_size=32, 
                    validation_split=0.2)

## 6. Evaluation

In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test)
print(f"Test Accuracy: {test_acc:.4f}")

y_pred = (model.predict(X_test) > 0.5).astype("int32")

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Not Drone', 'Drone']))

## 7. Save Model

In [ ]:
model.save('drone_sound_model.h5')
print("Model saved as drone_sound_model.h5")